# Step 5: Experiment Tracking with MLflow

**What we're doing:** Re-running our models, but this time logging every
run's hyperparameters, metrics, and the trained model artifact itself to
MLflow — so we can compare runs later without re-running anything.

**Why this matters:** In Step 4, if you changed `max_depth` and reran the
cell, your old results were just... gone, overwritten in memory. That's
fine for a notebook experiment, but it's exactly how real ML teams lose
track of 'wait, which run actually had the best AUC, and what params did
it use?' MLflow solves this by recording every run permanently.

**Colab note on MLflow's UI:** MLflow normally shows a web dashboard, but
Colab doesn't expose local ports directly. Two options:
1. Log everything in Colab, then `zip` and download the `mlruns/` folder,
   and run `mlflow ui` locally on your machine to view the dashboard.
2. Use `pyngrok` to tunnel the MLflow UI out of Colab (shown below, optional).

We'll do (1) as the primary path since it needs zero extra setup.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/churn-platform'

!pip install -q mlflow xgboost

import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"   # <-- new line

mlflow.set_tracking_uri(f'file:{PROJECT_ROOT}/mlruns')
mlflow.set_experiment('churn_prediction')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2026/06/30 14:31:51 INFO mlflow.tracking.fluent: Experiment with name 'churn_prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///content/drive/MyDrive/churn-platform/mlruns/370882125921401588', creation_time=1782829911340, effective_trace_archival_retention=None, experiment_id='370882125921401588', last_update_time=1782829911340, lifecycle_stage='active', name='churn_prediction', tags={}, trace_location=None, workspace='default'>

In [ ]:
df = pd.read_csv(f'{PROJECT_ROOT}/data/processed/churn_clean.csv')

X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']
categorical_cols = X.select_dtypes(include='object').columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

## 5.1 Helper function: train + log a run
Wrapping this once means every run logs consistently — same metric names,
same structure — which is what makes runs comparable later.

In [ ]:
import mlflow.xgboost

def train_and_log(model, model_name, params, X_train, y_train, X_test, y_test):
    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)

        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        metrics = {
            'accuracy': accuracy_score(y_test, preds),
            'precision': precision_score(y_test, preds),
            'recall': recall_score(y_test, preds),
            'f1': f1_score(y_test, preds),
            'auc': roc_auc_score(y_test, probs),
        }
        mlflow.log_metrics(metrics)

        # use the matching logger for the model type
        if isinstance(model, xgb.XGBClassifier):
            mlflow.xgboost.log_model(model, 'model')
        else:
            mlflow.sklearn.log_model(model, 'model')

        print(f'{model_name}: {metrics}')
        return metrics

## 5.2 Run several XGBoost configurations
We deliberately vary hyperparameters across runs — this is the actual
point of tracking: comparing many runs, not just logging one.

In [ ]:
xgb_configs = [
    {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05},
    {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.03},
]

for cfg in xgb_configs:
    model = xgb.XGBClassifier(**cfg, eval_metric='logloss', random_state=42)
    train_and_log(model, f"xgboost_depth{cfg['max_depth']}", cfg, X_train, y_train, X_test, y_test)

2026/06/30 14:39:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgboost_depth3: {'accuracy': 0.8005677785663591, 'precision': 0.6565656565656566, 'recall': 0.5213903743315508, 'f1': 0.5812220566318927, 'auc': np.float64(0.8446369061458576)}


2026/06/30 14:39:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgboost_depth4: {'accuracy': 0.7991483321504613, 'precision': 0.6511627906976745, 'recall': 0.5240641711229946, 'f1': 0.5807407407407408, 'auc': np.float64(0.8444018186984937)}


2026/06/30 14:39:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgboost_depth6: {'accuracy': 0.7970191625266146, 'precision': 0.6375, 'recall': 0.5454545454545454, 'f1': 0.5878962536023055, 'auc': np.float64(0.8413805574930895)}


In [ ]:
rf_configs = [
    {'n_estimators': 100, 'max_depth': 6},
    {'n_estimators': 300, 'max_depth': 8},
]

for cfg in rf_configs:
    model = RandomForestClassifier(**cfg, random_state=42, class_weight='balanced')
    train_and_log(model, f"random_forest_depth{cfg['max_depth']}", cfg, X_train, y_train, X_test, y_test)

2026/06/30 14:39:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


random_forest_depth6: {'accuracy': 0.7416607523066004, 'precision': 0.5084745762711864, 'recall': 0.8021390374331551, 'f1': 0.6224066390041494, 'auc': np.float64(0.8418416905629182)}


2026/06/30 14:40:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


random_forest_depth8: {'accuracy': 0.7601135557132718, 'precision': 0.5327272727272727, 'recall': 0.7834224598930482, 'f1': 0.6341991341991342, 'auc': np.float64(0.8436784727066057)}


## 5.3 Query all logged runs programmatically
This is the payoff: instead of remembering which run was best, we ask
MLflow directly.

In [ ]:
runs_df = mlflow.search_runs(experiment_names=['churn_prediction'])
cols_to_show = ['tags.mlflow.runName', 'metrics.auc', 'metrics.f1', 'metrics.precision', 'metrics.recall']
runs_df[cols_to_show].sort_values('metrics.auc', ascending=False)

,tags.mlflow.runName,metrics.auc,metrics.f1,metrics.precision,metrics.recall
4,xgboost_depth3,0.844637,0.581222,0.656566,0.521390
5,xgboost_depth3,0.844637,0.581222,0.656566,0.521390
3,xgboost_depth4,0.844402,0.580741,0.651163,0.524064
0,random_forest_depth8,0.843678,0.634199,0.532727,0.783422
1,random_forest_depth6,0.841842,0.622407,0.508475,0.802139
2,xgboost_depth6,0.841381,0.587896,0.637500,0.545455


## 5.4 (Optional) View the MLflow UI

**Locally (recommended):** download the `mlruns/` folder from your Drive,
then in a terminal:
```
cd churn-platform
mlflow ui --backend-store-uri file:./mlruns
```
Open http://localhost:5000 — you'll see a sortable, filterable table of
every run, with charts comparing metrics across runs. This is the artifact
worth screenshotting for your README.

In [ ]:
# OPTIONAL: tunnel the MLflow UI out of Colab using ngrok (needs a free
# ngrok account + authtoken). Skip this if you'd rather just view locally.

# !pip install -q pyngrok
# from pyngrok import ngrok
# import subprocess, time
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')
# subprocess.Popen(['mlflow', 'ui', '--backend-store-uri', f'file:{PROJECT_ROOT}/mlruns', '--port', '5000'])
# time.sleep(5)
# public_url = ngrok.connect(5000)
# print('MLflow UI:', public_url)